> ## SOLUTION / ANSWER KEY &mdash; Lab 7.8
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-08-retry-idempotency.ipynb`](../lab-08-retry-idempotency.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 7.8 &mdash; Retry & Idempotency

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Retry a flaky call, but cap the attempts so it can't loop forever
- Make sending idempotent with a key, so a re-run is safe
- See why idempotency matters most for money & messages

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps are **offline &amp; deterministic** (pure Python stdlib); the agent-assembly labs reuse the **LangChain-shaped shim** from Module 6. Advanced labs end with an **optional** cell that runs the **real** library. You are building the **email-drafting agent** &mdash; the client's Lab 4.1.

**Reference:** [Module 7 slides &mdash; Reliability: retry & idempotency](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-07-08"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
Models and tools are flaky, so wrap calls in a **retry** &mdash; but **cap** the attempts (deck slide
12). And design for **idempotency**: key each action so running the same task twice is **safe** and
never **double-sends** an email or **double-charges** a card. This is the subtlest and most important
discipline for anything that touches **money or messages**.

In [ ]:
# A helper that fails n_fail times, then succeeds -- to exercise the retry deterministically.
def flaky(n_fail):
    calls = {"n": 0}
    def f():
        calls["n"] += 1
        if calls["n"] <= n_fail:
            raise RuntimeError("transient")
        return "ok"
    return f
def raises(fn):
    try: fn(); return False
    except Exception: return True
print("helpers ready: flaky(n) fails n times then returns 'ok'")

## Your Turn
Complete `with_retry` (capped) and `send_once` (idempotent via a key set).

In [ ]:
def with_retry(fn, max_attempts=3):
    # call fn(); on failure retry up to max_attempts; raise the last error if all fail
    last = None
    for attempt in range(max_attempts):
        try:
            return fn()
        except Exception as e:
            last = e
    raise last

def send_once(key, sent):
    # idempotent: sending the same key twice must NOT double-send
    if key in sent:
        return "already_sent"
    sent.add(key)
    return "sent"

try:
    print('first try ok   :', with_retry(flaky(0)))
    print('after 2 fails  :', with_retry(flaky(2), 3))
    print('exhausted raises:', raises(lambda: with_retry(flaky(5), 3)))
    sent = set()
    print('send 4471 (1st):', send_once('4471', sent))
    print('send 4471 (2nd):', send_once('4471', sent))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("returns the result when the call succeeds", lambda: with_retry(flaky(0)) == "ok")
expect_true("succeeds after transient failures", lambda: with_retry(flaky(2), 3) == "ok")
expect_true("raises once attempts are exhausted", lambda: raises(lambda: with_retry(flaky(5), 3)))
expect_true("the first send goes out", lambda: send_once("4471", set()) == "sent")
expect_true("a duplicate send is suppressed (idempotent)", lambda: (lambda s: (send_once("4471", s), send_once("4471", s))[1])(set()) == "already_sent")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Retry with a cap, and key every irreversible action so a re-run is safe. Assume every step can fail -- and make failure safe and visible. Idempotency is what lets an automation run unattended.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>